In [66]:
import random as rd
from tqdm import tqdm
from typing import Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from sklearn.metrics import f1_score
from sklearn.dummy import DummyClassifier

import numpy as np
import pandas as pd

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import BPEDecoder, Replace

from transformers import PreTrainedTokenizerFast

In [16]:
dataset = load_dataset("mteb/emotion")
print(f"Dataset type: {type(dataset)}")
dataset

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 15956
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1988
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1986
    })
})

In [17]:
train_dataset = dataset["train"].to_pandas()
val_dataset = dataset["validation"].to_pandas()
test_dataset = dataset['test'].to_pandas()
print(f"Splits dtype: {type(train_dataset), type(val_dataset), type(train_dataset)}")
print(len(train_dataset), len(val_dataset), len(test_dataset))

Splits dtype: (<class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>)
15956 1988 1986


In [18]:
print("Random train row:")
print(train_dataset.sample(), end = '\n\n')
print("Random val row:")
print(val_dataset.sample(), end = '\n\n')
print("Random test row:")
print(test_dataset.sample())

Random train row:
                                                   text  label label_text
3378  i hear the word and i feel stronger and re ass...      1        joy

Random val row:
                                                   text  label label_text
1681  i was feeling extremely whiney and lonely and sad      0    sadness

Random test row:
                                                  text  label label_text
172  i have to admit i feel amused when i see the p...      1        joy


In [19]:
# Unique labels
train_dataset[["label", "label_text"]].drop_duplicates()

,label,label_text
0,0,sadness
2,3,anger
3,2,love
6,5,surprise
7,4,fear
8,1,joy


In [20]:
def get_label_text(label: int) -> str:
    label_data = train_dataset[["label", "label_text"]].drop_duplicates()
    for row in label_data.itertuples():
        if row.label == label:
            return row.label_text
    raise ValueError("There is no such label")

### Dummy model

In [67]:
dummy_clsf = DummyClassifier(strategy = 'most_frequent')
dummy_clsf

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None


In [72]:
dummy_clsf.fit(train_dataset["text"], train_dataset["label"])

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](6,)","[0.29,0.33,0.08,0.13,0.12,0.04]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](6,)","[0,1,2,3,4,5]"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,6
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [78]:
f1_score(y_pred = dummy_clsf.predict(train_dataset["text"]).tolist(),
         y_true = train_dataset["label"].tolist(), average = "macro")

0.0836423955056883

### Tokenization

In [ ]:
# Create tokeinzer object from HF tokenizers
tokenizer = Tokenizer(model = BPE(unk_token="[UNK]"))

# Trainer tokenizer
trainer = BpeTrainer(
    vocab_size =200,  # Desired vocabulary size
    min_frequency= 1,   # Minimum frequency for a token to be included
    special_tokens=["[PAD]", "[UNK]", "[CLS]"]
)
tokenizer.normalizer = Lowercase()
tokenizer.decoder = BPEDecoder(suffix = "#")

In [22]:
train_dataset["text"][:5]

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [23]:
tokenizer.train_from_iterator(train_dataset["text"], trainer = trainer)

In [24]:
vocab = tokenizer.get_vocab()
print(f"Length of vocabulary: {len(vocab)}")
vocab

Length of vocabulary: 200


{'ke ': 84,
 'c': 6,
 'be': 82,
 'or': 50,
 'o': 18,
 'ev': 116,
 'ch': 86,
 'ent': 182,
 'i feel ': 61,
 'ar': 60,
 'ed ': 52,
 'f ': 59,
 'ow': 79,
 'ough': 185,
 'es ': 104,
 'bu': 103,
 'an ': 111,
 'ri': 101,
 'are ': 194,
 'just ': 165,
 'for ': 115,
 'di': 110,
 'wa': 83,
 '[CLS]': 2,
 'o ': 43,
 'y ': 38,
 'f': 9,
 'om': 75,
 'no': 112,
 'as': 113,
 'st ': 196,
 'a ': 62,
 'ic': 155,
 'll ': 140,
 'i c': 158,
 'e': 8,
 'i have ': 198,
 'at ': 57,
 'ing': 42,
 'ch ': 146,
 'k ': 96,
 'al ': 181,
 'en': 48,
 'ow ': 109,
 'feeling ': 69,
 'and ': 51,
 'ra': 147,
 'k': 14,
 'le': 88,
 'si': 135,
 'el': 37,
 'li': 64,
 'con': 180,
 'it ': 87,
 'at': 91,
 'm ': 58,
 'r': 21,
 'on': 47,
 'v': 25,
 'feel': 41,
 'is ': 74,
 'ant ': 157,
 'e ': 30,
 'w': 26,
 'out ': 102,
 'da': 148,
 'h': 11,
 'z': 29,
 'wor': 189,
 'ed': 143,
 'i feel like ': 177,
 'because ': 195,
 'im ': 97,
 'it': 90,
 'p ': 117,
 'my ': 80,
 'd': 7,
 'when ': 170,
 've ': 72,
 'ld ': 128,
 'fu': 142,
 'the': 131,
 

In [25]:
# CLS token
vocab["[CLS]"]

2

In [26]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "[PAD]")

In [27]:
# Encoding with cls token
input_ids = main_tokenizer("Hi there! Never you mind that", padding = "max_length", max_length = 50, trucation = True)['input_ids']
input_ids

[11,
 35,
 32,
 45,
 8,
 1,
 3,
 17,
 116,
 67,
 167,
 16,
 33,
 34,
 32,
 91,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [28]:
main_tokenizer.decode(input_ids, skip_special_tokens = True)

'hi there never you mind that'

In [29]:
max_number_of_tokens = 0
for text in train_dataset["text"]:
    tokenized_text = main_tokenizer(text)['input_ids']
    max_number_of_tokens = max(max_number_of_tokens, len(tokenized_text))
print(f"Maxium number of tokens in a single sentence: {max_number_of_tokens}")

Maxium number of tokens in a single sentence: 160


In [30]:
# Create torch datasets
class EmotionDataset(Dataset):
    def __init__(self, data: pd.DataFrame, 
                 tokenizer: PreTrainedTokenizerFast, max_length: int = 160):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __getitem__(self, idx) -> Tuple[torch.tensor, int]:
        single_row = self.data.iloc[idx]
        raw_text = single_row["text"]
        tokenized_text = self.tokenizer(raw_text, padding = "max_length",
                                        max_length = self.max_length, truncation = True,
                                        return_tensors = "pt")["input_ids"]
        return tokenized_text.squeeze(dim = 0), single_row["label"].item()

    def __len__(self):
        return len(self.data)


In [31]:
# Random emotions dataset item
emotion_dataset = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
random_item = emotion_dataset[rd.randint(0, len(emotion_dataset) - 1)]
print(f"Random item tensor shape: {random_item[0].shape}")
random_item

Random item tensor shape: torch.Size([160])


(tensor([ 97,  69,  56,   5,  37,  64,  46,  36, 115,  54, 150,  84,  71,  82,
          44,  56,   5,  37,  64,  46,  22,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0]),
 3)

In [32]:
emotions_dataset_demo = EmotionDataset(train_dataset.iloc[:5, :], tokenizer = main_tokenizer)
print(len(emotions_dataset_demo))

5


In [33]:
# Prepare all the other datasets
emotions_dataset_train = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
emotions_dataset_val = EmotionDataset(val_dataset, tokenizer = main_tokenizer)
emotions_dataset_test = EmotionDataset(test_dataset, tokenizer = main_tokenizer)
print(f"Lengths of datasets: {len(emotions_dataset_train)}, {len(emotions_dataset_val)}, {len(emotions_dataset_test)}")


Lengths of datasets: 15956, 1988, 1986


In [34]:
# Create emotion model
class EmotionModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int,
                 embedding_num: int, pad_id: int = 0, num_layers: int = 1, num_cls: int = 6):
        super().__init__()
        self.embed = nn.Embedding(embedding_num, input_size, padding_idx = pad_id)
        self.linear = nn.Linear(hidden_size, num_cls)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first = False,
                          num_layers = num_layers, nonlinearity = "relu")
    
    def forward(self, x):
        y = self.embed(x)
        y = y.transpose(0, 1)
        out, h = self.rnn(y)
        y = h.squeeze(dim = 0)
        y = self.linear(y)
        return y

In [35]:
def inference(text: str, model: nn.Module) -> Tuple[int, str]:
    pass

In [36]:
@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 2e-3
    betas: Tuple[float, float] = (0.99, 0.98)
    epoch: int = 100

In [37]:
config = TrainConfig()
config

TrainConfig(batch_size=32, learning_rate=0.002, betas=(0.99, 0.98), epoch=100)

In [38]:
train_dataloader = DataLoader(dataset = emotions_dataset_train,
                              shuffle = True, batch_size = config.batch_size)

In [39]:
val_loader = DataLoader(dataset = emotions_dataset_val, shuffle = False, batch_size = config.batch_size)
val_loader

In [40]:
train_sample = next(iter(train_dataloader))
print(f"Shape tensor of train sample: {train_sample[0].shape}")

Shape tensor of train sample: torch.Size([32, 160])


In [41]:
len(main_tokenizer.get_vocab())

200

In [42]:
rnn_model = EmotionModel(input_size = 300,
                        hidden_size = 300,
                        embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')
rnn_model

EmotionModel(
  (embed): Embedding(200, 300, padding_idx=0)
  (linear): Linear(in_features=300, out_features=6, bias=True)
  (rnn): RNN(300, 300)
)

In [43]:
rnn_model(train_sample[0].to(device = 'cuda')).shape

torch.Size([32, 6])

In [44]:
# Training loop for model_training
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    save: bool = False,
    save_epoch: int = 0,
    base_path: str = '',
    save_model_name: str = '',
    save_optimizer_name : str = ''
) -> float:
    """Run one training epoch."""
    if save:
        torch.save(model.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_model_name)
        torch.save(optimizer.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_optimizer_name)
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for inputs, targets in tqdm(dataloader, desc="Epoch training"):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [45]:
example_input = emotions_dataset_train.tokenizer("The waiters at this restaurant are awful", return_tensors = "pt")["input_ids"]

In [46]:
# Given the model calculate f1-score on validation dataset
def calculate_val_score(dataloader: DataLoader,
                        model: nn.Module, device: str = "cuda") -> float:
    model.eval()
    y_preds = []
    y_trues = []
    for input_tensors, labels in dataloader:
        input_tensors, labels = input_tensors.to(device = device), labels.to(device = device)
        predictions = model(input_tensors)
        y_pred = predictions.softmax(dim = 1).argmax(dim = 1).detach().tolist()
        y_true = labels.detach().tolist()
        y_preds.extend(y_pred)
        y_trues.extend(y_true)
    return f1_score(y_true = y_trues, y_pred = y_preds, average = "macro")

### TF-IDF + Linear

In [ ]:
something = "tf-idf model"

### RRN training

In [47]:
rnn_model = EmotionModel(input_size = 300,
                         hidden_size = 150,
                         embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')

In [62]:
rnn_model.load_state_dict(torch.load("C:\\main\\GitHub\\nlpTasks\\artifacts\Epoch #2000-SimpleRNN.pth"))

<All keys matched successfully>

In [54]:
optimizer = optim.AdamW(params = rnn_model.parameters(), lr = config.learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index = 0)

In [61]:
optimizer.load_state_dict(torch.load("C:\\main\\GitHub\\nlpTasks\\artifacts\Epoch #2000-SimpleRNNOptimizer.pth"))

In [63]:
# Training looop 
for ep in tqdm(range(2_000), desc = "Current epoch"):
    if (ep+1) % 10 == 0:
        current_loss = train_one_epoch(model = rnn_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = True,
                        save_epoch = ep + 1,
                        base_path= "artifacts\\",
                        save_model_name="SimpleRNN.pth",
                        save_optimizer_name="SimpleRNNOptimizer.pth")
    else:
        current_loss = train_one_epoch(model = rnn_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = False) 
    current_val = calculate_val_score(val_loader, rnn_model, device = "cuda")
    print(f"Epoch # {ep + 1} | Train loss: {current_loss} | Val f1_score: {current_val}")
    

Current epoch:   0%|          | 1/2000 [00:07<4:04:40,  7.34s/it]

Epoch # 1 | Train loss: 0.6175923005552713 | Val f1_score: 0.2056486654536254


Current epoch:   0%|          | 2/2000 [00:15<4:18:13,  7.75s/it]

Epoch # 2 | Train loss: 0.4740601035137334 | Val f1_score: 0.20362184691544916


Current epoch:   0%|          | 3/2000 [00:23<4:25:15,  7.97s/it]

...

Epoch # 327 | Train loss: 0.45842279353457127 | Val f1_score: 0.20466285359032368


Current epoch:  16%|█▋        | 328/2000 [50:21<4:04:13,  8.76s/it]

Epoch # 328 | Train loss: 0.4584465801029024 | Val f1_score: 0.2027878096575122


Current epoch:  16%|█▋        | 328/2000 [50:24<4:16:58,  9.22s/it]


KeyboardInterrupt: 

### LSTM training

In [59]:
some_string = 'Here there is going to be LSTM training'

### GRU training

In [58]:
some_string = "Here is going to be GRU training"

### BERT training

In [60]:
some_string = "Here is going to be BERT training"